# Módulo 46 · DL II: redes convolucionales y el modelo de clase

Código del módulo para ejecutar en **Google Colab**, donde TensorFlow y PyTorch ya vienen instalados. Para entrenar más rápido activa la GPU en *Entorno de ejecución → Cambiar tipo de entorno de ejecución*. Las explicaciones, los ejercicios y las partes que funcionan en el navegador están en el módulo 46 de la web.

In [ ]:
# Solo si lo ejecutas en tu ordenador (en Colab ya está todo instalado):
# !pip install tensorflow matplotlib

### Keras: red densa frente a convolucional en MNIST (Colab)

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Sequential, layers

(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

def curvas(history, titulo):
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    for ax, m in zip(axs, ["loss", "accuracy"]):
        ax.plot(history.history[m], label="entrenamiento"); ax.plot(history.history["val_" + m], label="validación")
        ax.set_title(f"{m} ({titulo})"); ax.set_xlabel("Época"); ax.legend()
    plt.show()

# Red densa: cada imagen se aplana a 784 valores
model_dense = Sequential([layers.Input(shape=(784,)),
                          layers.Dense(50, activation="relu"), layers.Dense(50, activation="relu"),
                          layers.Dense(50, activation="relu"), layers.Dense(10, activation="softmax")])
model_dense.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = model_dense.fit(X_train.reshape(-1, 784), y_train, epochs=20, batch_size=30, validation_split=0.15, verbose=0)
curvas(h1, "densa")
print("Densa, accuracy en test:", model_dense.evaluate(X_test.reshape(-1, 784), y_test, verbose=0)[1])

# Red convolucional: las imágenes conservan su forma (28, 28, 1)
model_conv = Sequential([layers.Input(shape=(28, 28, 1)),
                         layers.Conv2D(32, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
                         layers.Conv2D(32, (3, 3), activation="relu"), layers.Conv2D(64, (3, 3), activation="relu"),
                         layers.Dropout(0.5), layers.Flatten(),
                         layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
model_conv.summary()
model_conv.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model_conv.fit(X_train[..., None], y_train, epochs=8, batch_size=32, validation_split=0.15, verbose=0)
curvas(h2, "convolucional")
print("Convolucional, accuracy en test:", model_conv.evaluate(X_test[..., None], y_test, verbose=0)[1])

model_conv.save("modelo_mnist_convol.keras")        # formato nativo de Keras 3 (.h5 es el formato antiguo)

### Keras: Fashion-MNIST con una red convolucional (Colab)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, layers
from tensorflow.keras.optimizers import Adam

(X_train, Y_train), (X_test, Y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train, X_test = X_train / 255, X_test / 255
clases = ["camiseta", "pantalón", "jersey", "vestido", "abrigo", "sandalia", "camisa", "zapatilla", "bolso", "botín"]
print(dict(zip(*np.unique(Y_train, return_counts=True))))        # 6000 de cada clase: equilibrado

model = Sequential([layers.Input(shape=(28, 28, 1)),
                    layers.Conv2D(32, (3, 3), activation="relu"), layers.MaxPooling2D((2, 2)),
                    layers.Conv2D(128, (3, 3), activation="relu"), layers.Flatten(),
                    layers.Dense(200, activation="relu"), layers.Dense(10, activation="softmax")])
model.compile(optimizer=Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X_train[..., None], Y_train, epochs=10, batch_size=30, validation_split=0.15, verbose=0)
print("Fashion-MNIST, accuracy en test:", model.evaluate(X_test[..., None], Y_test, verbose=0)[1])

### TensorBoard en Colab (sin montar Drive: los registros se guardan en la sesión)

In [ ]:
import datetime
import tensorflow as tf

log_dir = "logs/mnist/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)

%load_ext tensorboard
%tensorboard --logdir logs/mnist

model_conv.fit(X_train[..., None], y_train, epochs=15, batch_size=32, validation_split=0.15, callbacks=[tensorboard])